
# Enveda CASMI 2026 — Validation & Retrieval Baseline

This notebook reconstructs the validation work for the CASMI project after the previous VS Code session was lost.

## What this notebook does

1. Loads the training parquet.
2. Verifies the schema.
3. Creates a **molecule-disjoint validation query set**:
   - only molecules with at least 5 spectra
   - exactly one query spectrum per molecule
   - 5,000 queries
4. Keeps the remaining spectra as the reference library.
5. Detects exact duplicate spectra using a hash.
6. Converts common adduct/precursor combinations to an estimated **neutral molecular mass**.
7. Builds two candidate-retrieval routes:
   - strict same-adduct + same-polarity + precursor-m/z matching
   - neutral-mass matching across adducts **and polarity**
8. Unions those candidates.
9. Excludes exact-copy reference spectra for each query.
10. Measures candidate recall.
11. Runs a small spectral-similarity benchmark.
12. Computes Top-1, Top-5, Top-25 and MRR@25 using **all validation queries as the denominator**.

> Important: this is a local validation benchmark. It is not the final Kaggle submission pipeline.


In [2]:

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.home() / "Programming/ML/pytorch/Enveda_CASMI_26"
TRAIN_PATH = PROJECT_ROOT / "data/raw/train.parquet"

print("Project:", PROJECT_ROOT)
print("Train:", TRAIN_PATH)
print("Exists:", TRAIN_PATH.exists())


Project: /home/mardhav-rayapati/Programming/ML/pytorch/Enveda_CASMI_26
Train: /home/mardhav-rayapati/Programming/ML/pytorch/Enveda_CASMI_26/data/raw/train.parquet
Exists: True


## 1. Load training data

In [3]:

df = pd.read_parquet(TRAIN_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)


Shape: (2539608, 18)

Columns:
['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs', 'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity', 'collision_energy_ev', 'collision_energy_orig', 'collision_energy_orig_units']

Dtypes:
ingest_lib                         str
normalized_smiles                  str
inchikey                           str
inchikey14                         str
molecular_formula                  str
ionization_mode                    str
instrument_type                    str
adduct                             str
adduct_orig                        str
precursor_mz                   float64
precursor_error_ppm            float64
ms2_mzs                         object
ms2_normalized_intensities      object
num_peaks                        int64
base_peak_intensity            float64
collision_energy_ev             ob


## 2. Basic schema checks

Training data does **not** contain `spectrum_id`; the test set does.

For validation we therefore keep the original DataFrame index as the row identifier.


In [4]:
required_columns = {
    "inchikey14",
    "adduct",
    "ionization_mode",
    "precursor_mz",
    "ms2_mzs",
    "ms2_normalized_intensities",
}

missing = required_columns - set(df.columns)

assert not missing, f"Missing columns: {missing}"

assert df["inchikey14"].notna().all()
assert df["precursor_mz"].notna().all()

print("Schema checks passed.")

Schema checks passed.



## 3. Build a molecule-disjoint validation set

A molecule must have at least 5 spectra.

We sample exactly **one spectrum per molecule**, then sample up to 5,000 molecules.

This avoids putting another spectrum of the same molecule into the validation reference library merely because the query was sampled randomly.


In [5]:

molecule_counts = df.groupby("inchikey14").size().sort_values(ascending=False)

eligible_molecules = molecule_counts[molecule_counts >= 5].index

eligible_df = df[df["inchikey14"].isin(eligible_molecules)].copy()

validation_queries = (
    eligible_df
    .groupby("inchikey14", group_keys=False)
    .sample(n=1, random_state=42)
    .sample(n=min(5000, len(eligible_molecules)), random_state=42)
    .copy()
)

validation_indices = validation_queries.index

reference_df = df.loc[~df.index.isin(validation_indices)].copy()

print("Eligible molecules:", len(eligible_molecules))
print("Validation queries:", len(validation_queries))
print("Unique validation molecules:", validation_queries["inchikey14"].nunique())
print("Reference rows:", len(reference_df))
print("Index overlap:", len(set(validation_indices) & set(reference_df.index)))


Eligible molecules: 165492
Validation queries: 5000
Unique validation molecules: 5000
Reference rows: 2534608
Index overlap: 0


In [6]:

assert len(validation_queries) == validation_queries["inchikey14"].nunique()
assert len(set(validation_indices) & set(reference_df.index)) == 0

print("Validation split checks passed.")


Validation split checks passed.



## 4. Exact spectrum hashing

Some spectra are exact duplicates.

We do **not** delete validation queries. Instead, when retrieving candidates for a query, exact-copy reference spectra are excluded.

This prevents the validation benchmark from becoming a trivial lookup task.


In [7]:

def spectrum_hash(mzs, intensities):
    mzs = np.asarray(mzs, dtype=np.float32)
    intensities = np.asarray(intensities, dtype=np.float32)

    h = hashlib.blake2b(digest_size=16)
    h.update(mzs.tobytes())
    h.update(intensities.tobytes())

    return h.hexdigest()


validation_queries["spectrum_hash"] = [
    spectrum_hash(mz, inten)
    for mz, inten in zip(
        validation_queries["ms2_mzs"],
        validation_queries["ms2_normalized_intensities"],
    )
]

reference_df["spectrum_hash"] = [
    spectrum_hash(mz, inten)
    for mz, inten in zip(
        reference_df["ms2_mzs"],
        reference_df["ms2_normalized_intensities"],
    )
]

reference_hashes = set(reference_df["spectrum_hash"])

duplicate_query_mask = validation_queries["spectrum_hash"].isin(reference_hashes)

print("Validation queries with exact reference copies:",
      int(duplicate_query_mask.sum()))


Validation queries with exact reference copies: 79



## 5. Neutral-mass calculation

A major limitation of matching only the same adduct is that the same molecule can appear with different adducts.

For example:

- `[M+Na]+` and `[M-H]-` can represent the same neutral molecule.
- Their precursor m/z values are therefore not directly comparable.

We convert common adducts into an estimated neutral mass.

The mapping below covers the common adducts needed for the current baseline and several multiply charged/dimer forms.


In [8]:

ION_MASSES = {
    "H+": 1.007276466621,
    "Na+": 22.989218,
    "K+": 38.963158,
    "NH4+": 18.033823,
    "Cl-": 34.969402,
    "Li+": 7.015455,
    "Ca2+": 39.961493,
}

def neutral_mass_from_adduct(precursor_mz, adduct):
    mz = float(precursor_mz)

    if adduct == "[M+H]+":
        return mz - ION_MASSES["H+"]

    if adduct == "[M-H]-":
        return mz + ION_MASSES["H+"]

    if adduct == "[M+Na]+":
        return mz - ION_MASSES["Na+"]

    if adduct == "[M+K]+":
        return mz - ION_MASSES["K+"]

    if adduct == "[M+NH4]+":
        return mz - ION_MASSES["NH4+"]

    if adduct == "[M+Cl]-":
        return mz - ION_MASSES["Cl-"]

    if adduct == "[M+Li]+":
        return mz - ION_MASSES["Li+"]

    if adduct in ("[M]+", "[M]-"):
        return mz

    if adduct == "[2M+H]+":
        return (mz - ION_MASSES["H+"]) / 2

    if adduct == "[2M+Na]+":
        return (mz - ION_MASSES["Na+"]) / 2

    if adduct == "[2M+K]+":
        return (mz - ION_MASSES["K+"]) / 2

    if adduct == "[2M+NH4]+":
        return (mz - ION_MASSES["NH4+"]) / 2

    if adduct == "[M+2H]2+":
        return (mz * 2) - 2 * ION_MASSES["H+"]

    if adduct == "[M+3H]3+":
        return (mz * 3) - 3 * ION_MASSES["H+"]

    return np.nan


In [9]:

reference_df["neutral_mass"] = [
    neutral_mass_from_adduct(mz, adduct)
    for mz, adduct in zip(
        reference_df["precursor_mz"],
        reference_df["adduct"],
    )
]

validation_queries["neutral_mass"] = [
    neutral_mass_from_adduct(mz, adduct)
    for mz, adduct in zip(
        validation_queries["precursor_mz"],
        validation_queries["adduct"],
    )
]

print("Reference neutral-mass coverage:",
      reference_df["neutral_mass"].notna().mean())

print("Validation neutral-mass coverage:",
      validation_queries["neutral_mass"].notna().mean())

print("\nUnsupported / uncommon adducts:")
print(
    reference_df.loc[
        reference_df["neutral_mass"].isna(),
        "adduct"
    ].value_counts().head(20)
)


Reference neutral-mass coverage: 0.9300755777619261
Validation neutral-mass coverage: 0.9184

Unsupported / uncommon adducts:
adduct
[M+CH2O2-H]-      83011
[2M-H]-           55995
[M-H2O+H]+        21650
[M+C2H4O2-H]-      7426
[M-2H2O+H]+        3433
[2M+Na-2H]-        1430
[2M+CH2O2-H]-       664
[M-H2O]+            623
[M-CH3]-            618
[M-2H]-             454
[M-H2O-H]-          285
[M-H5O3]+           230
[2M+C2H4O2-H]-      187
[M+C2H4N]+          158
[M+2Na-H]+          110
[M-H2]2-            103
[M+Ca]2+             84
[M-C3H6NO2]-         72
[M+CH5O]+            66
[M-H2N]+             61
Name: count, dtype: int64



## 6. Candidate retrieval

There are two routes.

### Route A — direct matching

Require:

- same adduct
- same ionization mode
- precursor m/z within tolerance

### Route B — neutral-mass matching

Require:

- estimated neutral mass within tolerance

**No ionization-mode restriction is applied here.**

That is deliberate: the same molecule can occur as a positive ion in one spectrum and a negative ion in another.

The two candidate sets are then unioned.


In [10]:

def get_candidates_union(
    query,
    reference_df,
    precursor_tolerance=0.02,
    neutral_mass_tolerance=0.005,
):
    # Route A: strict direct matching
    direct_mask = (
        (reference_df["adduct"] == query["adduct"])
        & (reference_df["ionization_mode"] == query["ionization_mode"])
        & (
            np.abs(
                reference_df["precursor_mz"] - query["precursor_mz"]
            ) <= precursor_tolerance
        )
    )

    direct_candidates = reference_df.loc[direct_mask]

    # Route B: neutral mass matching across adducts AND polarity
    query_neutral_mass = query["neutral_mass"]

    if pd.notna(query_neutral_mass):
        neutral_mask = (
            reference_df["neutral_mass"].notna()
            & (
                np.abs(
                    reference_df["neutral_mass"] - query_neutral_mass
                ) <= neutral_mass_tolerance
            )
        )

        neutral_candidates = reference_df.loc[neutral_mask]
    else:
        neutral_candidates = reference_df.iloc[0:0]

    candidates = pd.concat(
        [direct_candidates, neutral_candidates]
    )

    candidates = candidates.loc[
        ~candidates.index.duplicated()
    ]

    # Remove exact-copy spectra from this query
    candidates = candidates[
        candidates["spectrum_hash"] != query["spectrum_hash"]
    ]

    return candidates


## 7. Check the two previously missing queries

In [11]:

# Reproduce the two cases that failed under strict same-adduct matching.
previous_missing_ids = {
    "RZNIFCWSNAETSG",
    "IKBBNHNPZIDCAB",
}

for _, query in validation_queries[
    validation_queries["inchikey14"].isin(previous_missing_ids)
].iterrows():

    candidates = get_candidates_union(
        query,
        reference_df,
        precursor_tolerance=0.02,
        neutral_mass_tolerance=0.005,
    )

    found = query["inchikey14"] in set(candidates["inchikey14"])

    print("=" * 70)
    print("Molecule:", query["inchikey14"])
    print("Adduct:", query["adduct"])
    print("Ionization:", query["ionization_mode"])
    print("Precursor:", query["precursor_mz"])
    print("Neutral mass:", query["neutral_mass"])
    print("Candidates:", len(candidates))
    print("True molecule found:", found)


Molecule: RZNIFCWSNAETSG
Adduct: [M+Na]+
Ionization: positive
Precursor: 358.1204
Neutral mass: 335.131182
Candidates: 1804
True molecule found: True
Molecule: IKBBNHNPZIDCAB
Adduct: [M-H]-
Ionization: negative
Precursor: 266.0589
Neutral mass: 267.066176466621
Candidates: 1051
True molecule found: True



## 8. Candidate recall over all 5,000 validation queries

Candidate recall answers:

> "Does the candidate-generation stage contain the true molecule?"

This must be checked before spending time optimizing spectral similarity.

If the true molecule is absent here, no downstream ranking model can recover it.


In [12]:

recall_records = []

for n, (_, query) in enumerate(validation_queries.iterrows(), start=1):
    candidates = get_candidates_union(
        query,
        reference_df,
        precursor_tolerance=0.02,
        neutral_mass_tolerance=0.005,
    )

    candidate_molecules = set(candidates["inchikey14"])

    recall_records.append({
        "query_index": query.name,
        "inchikey14": query["inchikey14"],
        "candidate_count": len(candidates),
        "found": query["inchikey14"] in candidate_molecules,
    })

    if n % 250 == 0:
        print(f"Processed {n}/{len(validation_queries)}")

candidate_recall_df = pd.DataFrame(recall_records)

candidate_recall = candidate_recall_df["found"].mean()

print("\nCandidate recall:", candidate_recall)
print("Found:", candidate_recall_df["found"].sum())
print("Missing:", (~candidate_recall_df["found"]).sum())
print("\nCandidate count:")
print(candidate_recall_df["candidate_count"].describe())


Processed 250/5000
Processed 500/5000
Processed 750/5000
Processed 1000/5000
Processed 1250/5000
Processed 1500/5000
Processed 1750/5000
Processed 2000/5000
Processed 2250/5000
Processed 2500/5000
Processed 2750/5000
Processed 3000/5000
Processed 3250/5000
Processed 3500/5000
Processed 3750/5000
Processed 4000/5000
Processed 4250/5000
Processed 4500/5000
Processed 4750/5000
Processed 5000/5000

Candidate recall: 0.9942
Found: 4971
Missing: 29

Candidate count:
count    5000.000000
mean     1382.867400
std      1246.604716
min         0.000000
25%       348.000000
50%      1037.500000
75%      2110.250000
max      5536.000000
Name: candidate_count, dtype: float64



## 9. Spectral cosine similarity

This is the baseline spectrum-to-spectrum similarity:

- intensity-weighted cosine similarity
- one-to-one peak matching
- configurable m/z tolerance
- minimum intensity filtering

The current implementation is intentionally simple and will be optimized later.


In [13]:

def filter_peaks(mz, intensity, min_intensity=0.01):
    mz = np.asarray(mz)
    intensity = np.asarray(intensity)

    mask = intensity >= min_intensity

    return mz[mask], intensity[mask]


def spectral_cosine_similarity(
    mz1,
    intensity1,
    mz2,
    intensity2,
    tolerance=0.01,
    min_intensity=0.01,
):
    mz1, intensity1 = filter_peaks(
        mz1, intensity1, min_intensity
    )
    mz2, intensity2 = filter_peaks(
        mz2, intensity2, min_intensity
    )

    possible_matches = []

    for i in range(len(mz1)):
        differences = np.abs(mz2 - mz1[i])

        for j in np.where(differences <= tolerance)[0]:
            possible_matches.append(
                (differences[j], i, j)
            )

    possible_matches.sort(key=lambda x: x[0])

    used_1 = set()
    used_2 = set()

    matched_pairs = []

    for difference, i, j in possible_matches:
        if i in used_1 or j in used_2:
            continue

        used_1.add(i)
        used_2.add(j)

        matched_pairs.append((i, j))

    numerator = sum(
        intensity1[i] * intensity2[j]
        for i, j in matched_pairs
    )

    denominator = np.sqrt(
        np.sum(intensity1 ** 2)
        * np.sum(intensity2 ** 2)
    )

    if denominator == 0:
        return 0.0

    return numerator / denominator



## 10. Retrieval and molecule-level aggregation

A query can match many spectra belonging to the same molecule.

We therefore aggregate spectrum-level evidence to the molecule level.

Current baseline:

> **maximum spectrum similarity per molecule**

We also keep mean similarity and number of matching spectra for inspection.


In [14]:

def retrieve_molecules(
    query,
    reference_df,
    precursor_tolerance=0.02,
    neutral_mass_tolerance=0.005,
    tolerance=0.01,
    min_intensity=0.01,
):
    candidates = get_candidates_union(
        query,
        reference_df,
        precursor_tolerance=precursor_tolerance,
        neutral_mass_tolerance=neutral_mass_tolerance,
    )

    if len(candidates) == 0:
        return pd.DataFrame(
            columns=[
                "inchikey14",
                "max_score",
                "mean_score",
                "num_spectra",
                "molecule_rank",
            ]
        )

    results = []

    for idx, candidate in candidates.iterrows():
        score = spectral_cosine_similarity(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            candidate["ms2_mzs"],
            candidate["ms2_normalized_intensities"],
            tolerance=tolerance,
            min_intensity=min_intensity,
        )

        results.append({
            "inchikey14": candidate["inchikey14"],
            "score": score,
        })

    results_df = pd.DataFrame(results)

    molecule_results = (
        results_df
        .groupby("inchikey14")
        .agg(
            max_score=("score", "max"),
            mean_score=("score", "mean"),
            num_spectra=("score", "count"),
        )
        .sort_values("max_score", ascending=False)
        .reset_index()
    )

    molecule_results["molecule_rank"] = (
        np.arange(len(molecule_results)) + 1
    )

    return molecule_results



## 11. Small benchmark

Run a small benchmark first.

This is intentionally separate from the full 5,000-query benchmark because the current Python implementation of spectral similarity is computationally expensive.

The evaluator uses **all queries as the denominator**:

- missing rank → Top-k failure
- missing rank → MRR contribution 0


In [15]:

def evaluate_ranks(ranks):
    ranks = pd.Series(ranks, dtype="float64")

    top1 = (ranks <= 1).fillna(False).mean()
    top5 = (ranks <= 5).fillna(False).mean()
    top25 = (ranks <= 25).fillna(False).mean()

    reciprocal = np.where(
        ranks.notna() & (ranks <= 25),
        1.0 / ranks,
        0.0,
    )

    mrr25 = reciprocal.mean()

    return {
        "Top-1": top1,
        "Top-5": top5,
        "Top-25": top25,
        "MRR@25": mrr25,
    }


In [16]:

N_BENCHMARK = 50

benchmark_queries = validation_queries.iloc[:N_BENCHMARK].copy()

benchmark_records = []

for n, (_, query) in enumerate(
    benchmark_queries.iterrows(),
    start=1,
):
    results = retrieve_molecules(
        query,
        reference_df,
        precursor_tolerance=0.02,
        neutral_mass_tolerance=0.005,
        tolerance=0.01,
        min_intensity=0.01,
    )

    if len(results) == 0:
        rank = np.nan
        candidate_count = 0
    else:
        matches = results.index[
            results["inchikey14"] == query["inchikey14"]
        ]

        if len(matches):
            rank = results.loc[matches[0], "molecule_rank"]
        else:
            rank = np.nan

        candidate_count = len(results)

    benchmark_records.append({
        "query_index": query.name,
        "inchikey14": query["inchikey14"],
        "candidate_molecules": candidate_count,
        "rank": rank,
    })

    print(
        f"{n:>3}/{N_BENCHMARK} | "
        f"candidate molecules={candidate_count:>5} | "
        f"rank={rank}"
    )

benchmark_df = pd.DataFrame(benchmark_records)

metrics = evaluate_ranks(benchmark_df["rank"])

print("\n" + "=" * 50)
print("BENCHMARK RESULTS")
print("=" * 50)

for name, value in metrics.items():
    print(f"{name:8s}: {value:.4f}")

print("\nNot found:",
      benchmark_df["rank"].isna().sum(),
      "/",
      len(benchmark_df))


  1/50 | candidate molecules=   42 | rank=6
  2/50 | candidate molecules=   31 | rank=1
  3/50 | candidate molecules=  154 | rank=1
  4/50 | candidate molecules=  225 | rank=1
  5/50 | candidate molecules=  167 | rank=1
  6/50 | candidate molecules=  262 | rank=28
  7/50 | candidate molecules=  651 | rank=1
  8/50 | candidate molecules=  322 | rank=1
  9/50 | candidate molecules=  861 | rank=1
 10/50 | candidate molecules=   34 | rank=2
 11/50 | candidate molecules=   41 | rank=5
 12/50 | candidate molecules=  279 | rank=159
 13/50 | candidate molecules=   86 | rank=1
 14/50 | candidate molecules=  622 | rank=1
 15/50 | candidate molecules=  325 | rank=1
 16/50 | candidate molecules=  245 | rank=1
 17/50 | candidate molecules=  400 | rank=2
 18/50 | candidate molecules=  280 | rank=1
 19/50 | candidate molecules=  130 | rank=5
 20/50 | candidate molecules=   74 | rank=3
 21/50 | candidate molecules=  882 | rank=2
 22/50 | candidate molecules=  389 | rank=1
 23/50 | candidate molecules=


## 12. Inspect benchmark results

The rank distribution is useful for seeing whether failures are mostly candidate-generation failures or ranking failures.


In [17]:

benchmark_df.sort_values(
    "rank",
    na_position="last"
).head(25)


,query_index,inchikey14,candidate_molecules,rank
1,499628,LKEHPZAKDODNJF,31,1
2,2013348,UCNNCRSNVCNWJW,154,1
3,648414,OMZVBSDHVZLRQF,225,1
4,137556,CVRQYKRJJIPBAS,167,1
7,682537,PFGZANZPSCXOGE,322,1
6,390477,JCRONLIMSZZIKP,651,1
14,732354,QGNXJDQSBDSNQO,325,1
8,215707,FMEILJGGJVHEAP,861,1
15,1046256,XQTYFHLKAUJCME,245,1
12,1405209,HOYCASTVMCEOTP,86,1



## 13. Important next steps

Once this notebook is working again:

1. **Do not optimize the neural model yet.**
2. Measure candidate recall carefully.
3. Optimize spectral similarity using a `searchsorted`-based implementation.
4. Re-run the 50-query benchmark.
5. Test 500 queries.
6. Only then run the full 5,000-query benchmark.
7. Compare molecule aggregation strategies:
   - max score
   - mean of top-k spectrum scores
   - weighted/top-k aggregation
8. Improve neutral-mass/adduct parsing to cover more of the adduct vocabulary.
9. Eventually build the final top-25 SMILES submission pipeline.

### Current baseline concepts

- Validation is molecule-disjoint.
- Exact duplicate reference spectra are excluded per query.
- Direct matching is strict.
- Neutral-mass matching crosses adducts and polarity.
- Molecule ranking currently uses maximum spectral cosine similarity.
- MRR@25 uses every query in the denominator.

This notebook is intended as the clean restart point after the VS Code crash.


In [18]:
# Inspect the queries where the true molecule was not Top-5

failed = benchmark_df[
    benchmark_df["rank"].isna() | (benchmark_df["rank"] > 5)
].copy()

failed.sort_values("rank", na_position="last")

,query_index,inchikey14,candidate_molecules,rank
0,1754474,HMXYYLCNLNJVPN,42,6
5,2124935,ZGWMJBPVGJBFGH,262,28
37,356459,IKBBNHNPZIDCAB,68,45
42,629765,OCZSTDMRORZACG,219,55
11,813821,RZNIFCWSNAETSG,279,159


In [19]:
def inspect_query(query_index):
    query = validation_queries.loc[query_index]

    print("=" * 80)
    print("QUERY")
    print("=" * 80)

    print("Index:", query_index)
    print("Molecule:", query["inchikey14"])
    print("Adduct:", query["adduct"])
    print("Ionization:", query["ionization_mode"])
    print("Precursor m/z:", query["precursor_mz"])
    print("Neutral mass:", query["neutral_mass"])

    candidates = get_candidates_union(
        query,
        reference_df,
        precursor_tolerance=0.02,
        neutral_mass_tolerance=0.005,
    )

    print("\nCandidate spectra:", len(candidates))
    print(
        "Candidate molecules:",
        candidates["inchikey14"].nunique()
    )

    results = []

    for idx, candidate in candidates.iterrows():

        score = spectral_cosine_similarity(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            candidate["ms2_mzs"],
            candidate["ms2_normalized_intensities"],
            tolerance=0.01,
            min_intensity=0.01,
        )

        results.append({
            "reference_index": idx,
            "inchikey14": candidate["inchikey14"],
            "score": score,
            "adduct": candidate["adduct"],
            "ionization_mode": candidate["ionization_mode"],
            "precursor_mz": candidate["precursor_mz"],
        })

    results = pd.DataFrame(results)

    molecule_results = (
        results
        .groupby("inchikey14")
        .agg(
            max_score=("score", "max"),
            mean_score=("score", "mean"),
            num_spectra=("score", "count"),
        )
        .sort_values("max_score", ascending=False)
        .reset_index()
    )

    molecule_results["rank"] = (
        np.arange(len(molecule_results)) + 1
    )

    true_result = molecule_results[
        molecule_results["inchikey14"] == query["inchikey14"]
    ]

    print("\nTRUE MOLECULE:")
    print(true_result)

    print("\nTOP 20:")
    display(molecule_results.head(20))

    return query, results, molecule_results

In [20]:
inspect_query(
    benchmark_df.sort_values(
        "rank",
        na_position="last"
    ).iloc[0]["query_index"]
)

QUERY
Index: 499628
Molecule: LKEHPZAKDODNJF
Adduct: [2M-H]-
Ionization: negative
Precursor m/z: 681.2043
Neutral mass: nan

Candidate spectra: 112
Candidate molecules: 31

TRUE MOLECULE:
       inchikey14  max_score  mean_score  num_spectra  rank
0  LKEHPZAKDODNJF   0.995495    0.934247            3     1

TOP 20:


,inchikey14,max_score,mean_score,num_spectra,rank
0,LKEHPZAKDODNJF,0.995495,0.934247,3,1
1,FZQMSIRNUNCOMJ,0.993289,0.940150,4,2
2,MKEXNAFYNBMSHO,0.991997,0.829061,4,3
3,CSRVNEPGDALWGI,0.991466,0.618973,4,4
4,LIMGJIUWJAYOSZ,0.991341,0.726960,4,5
5,JWHRMHYEEFMMTE,0.990505,0.871050,4,6
6,UUIJHQVIMWVTRG,0.989979,0.866661,4,7
7,KYANOAXTNTYOEB,0.989851,0.967983,3,8
8,FWCXTARXPLRKDM,0.989447,0.988840,3,9
9,JMDPDYIWHFIRLK,0.989391,0.875149,4,10


(ingest_lib                                                            enveda-180
 normalized_smiles                       CCc1cc2c(NCCc3nnc4c(=O)[nH]ccn34)ncnc2s1
 inchikey                                             LKEHPZAKDODNJF-UHFFFAOYSA-N
 inchikey14                                                        LKEHPZAKDODNJF
 molecular_formula                                                     C15H15N7OS
 ionization_mode                                                         negative
 instrument_type                                                          timsTOF
 adduct                                                                   [2M-H]-
 adduct_orig                                                              [2M-H]-
 precursor_mz                                                            681.2043
 precursor_error_ppm                                                     0.536426
 ms2_mzs                        [65.9981, 70.3567, 106.0418, 108.0202, 109.028...
 ms2_normalized_

In [21]:
def spectral_cosine_similarity_diagnostic(
    mz1,
    intensity1,
    mz2,
    intensity2,
    tolerance=0.01,
    min_intensity=0.01,
):
    mz1, intensity1 = filter_peaks(
        mz1, intensity1, min_intensity
    )
    mz2, intensity2 = filter_peaks(
        mz2, intensity2, min_intensity
    )

    possible_matches = []

    for i in range(len(mz1)):
        differences = np.abs(mz2 - mz1[i])

        for j in np.where(differences <= tolerance)[0]:
            possible_matches.append(
                (differences[j], i, j)
            )

    possible_matches.sort(key=lambda x: x[0])

    used_1 = set()
    used_2 = set()
    matched_pairs = []

    for difference, i, j in possible_matches:
        if i in used_1 or j in used_2:
            continue

        used_1.add(i)
        used_2.add(j)

        matched_pairs.append((i, j))

    numerator = sum(
        intensity1[i] * intensity2[j]
        for i, j in matched_pairs
    )

    denominator = np.sqrt(
        np.sum(intensity1 ** 2)
        * np.sum(intensity2 ** 2)
    )

    score = 0.0 if denominator == 0 else numerator / denominator

    return {
        "score": score,
        "matched_peaks": len(matched_pairs),
        "query_peaks": len(mz1),
        "reference_peaks": len(mz2),
    }

In [22]:
query, results, molecule_results = inspect_query(499628)

QUERY
Index: 499628
Molecule: LKEHPZAKDODNJF
Adduct: [2M-H]-
Ionization: negative
Precursor m/z: 681.2043
Neutral mass: nan

Candidate spectra: 112
Candidate molecules: 31

TRUE MOLECULE:
       inchikey14  max_score  mean_score  num_spectra  rank
0  LKEHPZAKDODNJF   0.995495    0.934247            3     1

TOP 20:


,inchikey14,max_score,mean_score,num_spectra,rank
0,LKEHPZAKDODNJF,0.995495,0.934247,3,1
1,FZQMSIRNUNCOMJ,0.993289,0.940150,4,2
2,MKEXNAFYNBMSHO,0.991997,0.829061,4,3
3,CSRVNEPGDALWGI,0.991466,0.618973,4,4
4,LIMGJIUWJAYOSZ,0.991341,0.726960,4,5
5,JWHRMHYEEFMMTE,0.990505,0.871050,4,6
6,UUIJHQVIMWVTRG,0.989979,0.866661,4,7
7,KYANOAXTNTYOEB,0.989851,0.967983,3,8
8,FWCXTARXPLRKDM,0.989447,0.988840,3,9
9,JMDPDYIWHFIRLK,0.989391,0.875149,4,10


In [23]:
true_ref = results[
    results["inchikey14"] == query["inchikey14"]
].sort_values("score", ascending=False).iloc[0]

wrong_ref = results[
    results["inchikey14"] != query["inchikey14"]
].sort_values("score", ascending=False).iloc[0]

print("TRUE reference:")
print(true_ref)

print("\nWRONG reference:")
print(wrong_ref)

TRUE reference:
reference_index            499629
inchikey14         LKEHPZAKDODNJF
score                    0.995495
adduct                    [2M-H]-
ionization_mode          negative
precursor_mz             681.2043
Name: 65, dtype: object

WRONG reference:
reference_index            240649
inchikey14         FZQMSIRNUNCOMJ
score                    0.993289
adduct                    [2M-H]-
ionization_mode          negative
precursor_mz             681.2042
Name: 26, dtype: object


In [24]:
for label, row in [
    ("TRUE", true_ref),
    ("WRONG", wrong_ref),
]:
    ref = reference_df.loc[row["reference_index"]]

    diagnostic = spectral_cosine_similarity_diagnostic(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        ref["ms2_mzs"],
        ref["ms2_normalized_intensities"],
    )

    print("\n" + "=" * 50)
    print(label)
    print("=" * 50)

    print("Molecule:", row["inchikey14"])
    print("Score:", diagnostic["score"])
    print("Query peaks:", diagnostic["query_peaks"])
    print("Reference peaks:", diagnostic["reference_peaks"])
    print("Matched peaks:", diagnostic["matched_peaks"])


TRUE
Molecule: LKEHPZAKDODNJF
Score: 0.9954949890073942
Query peaks: 4
Reference peaks: 3
Matched peaks: 3

WRONG
Molecule: FZQMSIRNUNCOMJ
Score: 0.9932892091488834
Query peaks: 4
Reference peaks: 6
Matched peaks: 4


In [25]:
query = validation_queries.loc[499628]

print("Original number of peaks:", query["num_peaks"])

print("\nQuery m/z:")
print(query["ms2_mzs"])

print("\nQuery normalized intensities:")
print(query["ms2_normalized_intensities"])

Original number of peaks: 104

Query m/z:
[ 65.9981  70.3567 106.0418 108.0202 109.0287 122.037  149.047  149.0589
 149.0706 149.0783 150.0518 161.0466 178.0472 186.4864 190.0301 196.6187
 197.6883 205.0603 205.096  205.1027 224.5158 235.6777 235.8021 247.9535
 312.7274 325.1429 325.1653 333.0924 338.9061 339.2818 339.7124 339.8244
 339.922  339.9881 340.011  340.0312 340.0993 340.1472 340.1576 340.1633
 340.1777 340.1834 340.2035 340.215  340.2495 340.2955 340.3759 340.4334
 340.4622 340.5024 340.5197 340.5484 340.5829 340.6059 340.6117 340.6309
 340.6519 340.7497 340.7899 340.8129 340.8216 340.8589 340.8676 340.8842
 340.8992 340.905  340.928  340.9495 340.9625 340.9855 340.997  341.02
 341.043  341.0661 341.0776 341.1001 341.1238 341.1374 341.1466 341.1581
 341.1984 341.2617 341.3078 341.3768 341.3999 341.5495 341.6877 341.7338
 342.0795 342.0939 342.7828 342.8175 343.5302 343.5909 345.3256 345.3748
 345.7947 349.2446 349.3815 350.0253 355.0737 369.1937 397.6065 444.8549]

Query nor

In [26]:
for label, row in [
    ("TRUE", true_ref),
    ("WRONG", wrong_ref),
]:
    ref = reference_df.loc[row["reference_index"]]

    mz, intensity = filter_peaks(
        ref["ms2_mzs"],
        ref["ms2_normalized_intensities"],
        min_intensity=0.01,
    )

    print("\n" + "=" * 60)
    print(label)
    print("Molecule:", row["inchikey14"])
    print("Original peaks:", ref["num_peaks"])
    print("Peaks after filtering:", len(mz))

    print("\nm/z:")
    print(mz)

    print("\nIntensity:")
    print(intensity)


TRUE
Molecule: LKEHPZAKDODNJF
Original peaks: 51
Peaks after filtering: 3

m/z:
[149.0464 340.0989 341.0997]

Intensity:
[0.03010489 1.         0.08423012]

WRONG
Molecule: FZQMSIRNUNCOMJ
Original peaks: 39
Peaks after filtering: 6

m/z:
[108.017  149.0466 161.0487 289.1645 340.098  341.0998]

Intensity:
[0.01703775 0.13562365 0.0113319  0.02154656 1.         0.18777432]


In [27]:
thresholds = [0.0, 0.001, 0.002, 0.005, 0.01, 0.02]

for threshold in thresholds:

    true_score = spectral_cosine_similarity(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        reference_df.loc[
            true_ref["reference_index"],
            "ms2_mzs"
        ],
        reference_df.loc[
            true_ref["reference_index"],
            "ms2_normalized_intensities"
        ],
        tolerance=0.01,
        min_intensity=threshold,
    )

    wrong_score = spectral_cosine_similarity(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        reference_df.loc[
            wrong_ref["reference_index"],
            "ms2_mzs"
        ],
        reference_df.loc[
            wrong_ref["reference_index"],
            "ms2_normalized_intensities"
        ],
        tolerance=0.01,
        min_intensity=threshold,
    )

    q_peaks = np.sum(
        np.asarray(query["ms2_normalized_intensities"]) >= threshold
    )

    print(
        f"threshold={threshold:>6} | "
        f"query peaks={q_peaks:>3} | "
        f"true={true_score:.6f} | "
        f"wrong={wrong_score:.6f}"
    )

threshold=   0.0 | query peaks=104 | true=0.995579 | wrong=0.993165
threshold= 0.001 | query peaks= 19 | true=0.995579 | wrong=0.993175
threshold= 0.002 | query peaks=  8 | true=0.995571 | wrong=0.993177
threshold= 0.005 | query peaks=  5 | true=0.995605 | wrong=0.993230
threshold=  0.01 | query peaks=  4 | true=0.995495 | wrong=0.993289
threshold=  0.02 | query peaks=  4 | true=0.995495 | wrong=0.993097


In [28]:
def spectral_cosine_sqrt_intensity(
    mz1,
    intensity1,
    mz2,
    intensity2,
    tolerance=0.01,
    min_intensity=0.001,
):
    mz1 = np.asarray(mz1)
    intensity1 = np.asarray(intensity1)

    mz2 = np.asarray(mz2)
    intensity2 = np.asarray(intensity2)

    mask1 = intensity1 >= min_intensity
    mask2 = intensity2 >= min_intensity

    mz1 = mz1[mask1]
    intensity1 = np.sqrt(intensity1[mask1])

    mz2 = mz2[mask2]
    intensity2 = np.sqrt(intensity2[mask2])

    possible_matches = []

    for i in range(len(mz1)):
        differences = np.abs(mz2 - mz1[i])

        for j in np.where(differences <= tolerance)[0]:
            possible_matches.append(
                (differences[j], i, j)
            )

    possible_matches.sort(key=lambda x: x[0])

    used_1 = set()
    used_2 = set()

    matched_pairs = []

    for difference, i, j in possible_matches:
        if i in used_1 or j in used_2:
            continue

        used_1.add(i)
        used_2.add(j)

        matched_pairs.append((i, j))

    numerator = sum(
        intensity1[i] * intensity2[j]
        for i, j in matched_pairs
    )

    denominator = np.sqrt(
        np.sum(intensity1 ** 2)
        * np.sum(intensity2 ** 2)
    )

    if denominator == 0:
        return 0.0

    return numerator / denominator

In [29]:
for label, row in [
    ("TRUE", true_ref),
    ("WRONG", wrong_ref),
]:
    ref = reference_df.loc[row["reference_index"]]

    score = spectral_cosine_sqrt_intensity(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        ref["ms2_mzs"],
        ref["ms2_normalized_intensities"],
    )

    print(label, score)

TRUE 0.9707478774361344
WRONG 0.9586613647841575


In [30]:
def top_n_peaks(mz, intensity, n=20):
    mz = np.asarray(mz)
    intensity = np.asarray(intensity)

    n = min(n, len(mz))

    indices = np.argsort(intensity)[-n:]

    # Put them back in m/z order
    indices = indices[np.argsort(mz[indices])]

    return mz[indices], intensity[indices]

In [31]:
for n in [5, 10, 20, 30, 50, 100]:

    q_mz, q_int = top_n_peaks(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        n
    )

    t_ref = reference_df.loc[
        true_ref["reference_index"]
    ]

    w_ref = reference_df.loc[
        wrong_ref["reference_index"]
    ]

    true_score = spectral_cosine_sqrt_intensity(
        q_mz,
        q_int,
        t_ref["ms2_mzs"],
        t_ref["ms2_normalized_intensities"],
        tolerance=0.01,
        min_intensity=0.0,
    )

    wrong_score = spectral_cosine_sqrt_intensity(
        q_mz,
        q_int,
        w_ref["ms2_mzs"],
        w_ref["ms2_normalized_intensities"],
        tolerance=0.01,
        min_intensity=0.0,
    )

    print(
        f"Top {n:3d} | "
        f"true={true_score:.6f} | "
        f"wrong={wrong_score:.6f} | "
        f"diff={true_score-wrong_score:.6f}"
    )

Top   5 | true=0.957692 | wrong=0.958063 | diff=-0.000371
Top  10 | true=0.961318 | wrong=0.955212 | diff=0.006106
Top  20 | true=0.965828 | wrong=0.954108 | diff=0.011720
Top  30 | true=0.966968 | wrong=0.950956 | diff=0.016012
Top  50 | true=0.966766 | wrong=0.947896 | diff=0.018870
Top 100 | true=0.967392 | wrong=0.943867 | diff=0.023525


In [32]:
def spectral_cosine_top_n(
    mz1,
    intensity1,
    mz2,
    intensity2,
    top_n=None,
    tolerance=0.01,
):
    mz1 = np.asarray(mz1)
    intensity1 = np.asarray(intensity1)

    mz2 = np.asarray(mz2)
    intensity2 = np.asarray(intensity2)

    # Keep the strongest N query peaks
    if top_n is not None and len(mz1) > top_n:
        idx = np.argsort(intensity1)[-top_n:]
        mz1 = mz1[idx]
        intensity1 = intensity1[idx]

    # Keep the strongest N reference peaks too
    if top_n is not None and len(mz2) > top_n:
        idx = np.argsort(intensity2)[-top_n:]
        mz2 = mz2[idx]
        intensity2 = intensity2[idx]

    # sqrt intensity weighting
    intensity1 = np.sqrt(intensity1)
    intensity2 = np.sqrt(intensity2)

    possible_matches = []

    for i in range(len(mz1)):
        differences = np.abs(mz2 - mz1[i])

        for j in np.where(differences <= tolerance)[0]:
            possible_matches.append(
                (differences[j], i, j)
            )

    possible_matches.sort(key=lambda x: x[0])

    used_1 = set()
    used_2 = set()
    matched_pairs = []

    for difference, i, j in possible_matches:
        if i in used_1 or j in used_2:
            continue

        used_1.add(i)
        used_2.add(j)

        matched_pairs.append((i, j))

    numerator = sum(
        intensity1[i] * intensity2[j]
        for i, j in matched_pairs
    )

    denominator = np.sqrt(
        np.sum(intensity1 ** 2)
        * np.sum(intensity2 ** 2)
    )

    if denominator == 0:
        return 0.0

    return numerator / denominator

In [33]:
def retrieve_molecules_top_n(
    query,
    reference_df,
    top_n=None,
    precursor_tolerance=0.02,
    neutral_mass_tolerance=0.005,
    tolerance=0.01,
):
    candidates = get_candidates_union(
        query,
        reference_df,
        precursor_tolerance=precursor_tolerance,
        neutral_mass_tolerance=neutral_mass_tolerance,
    )

    if len(candidates) == 0:
        return pd.DataFrame(
            columns=[
                "inchikey14",
                "max_score",
                "mean_score",
                "num_spectra",
                "molecule_rank",
            ]
        )

    results = []

    for idx, candidate in candidates.iterrows():

        score = spectral_cosine_top_n(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            candidate["ms2_mzs"],
            candidate["ms2_normalized_intensities"],
            top_n=top_n,
            tolerance=tolerance,
        )

        results.append({
            "inchikey14": candidate["inchikey14"],
            "score": score,
        })

    results_df = pd.DataFrame(results)

    molecule_results = (
        results_df
        .groupby("inchikey14")
        .agg(
            max_score=("score", "max"),
            mean_score=("score", "mean"),
            num_spectra=("score", "count"),
        )
        .sort_values("max_score", ascending=False)
        .reset_index()
    )

    molecule_results["molecule_rank"] = (
        np.arange(len(molecule_results)) + 1
    )

    return molecule_results

In [34]:
top_n_values = [10, 20, 30, 50, None]

all_results = []

for top_n in top_n_values:

    print("=" * 60)
    print("TOP N:", top_n)
    print("=" * 60)

    ranks = []

    for n, (_, query) in enumerate(
        benchmark_queries.iterrows(),
        start=1
    ):

        results = retrieve_molecules_top_n(
            query,
            reference_df,
            top_n=top_n,
            precursor_tolerance=0.02,
            neutral_mass_tolerance=0.005,
            tolerance=0.01,
        )

        if len(results) == 0:
            rank = np.nan

        else:
            matches = results.index[
                results["inchikey14"] == query["inchikey14"]
            ]

            if len(matches):
                rank = results.loc[
                    matches[0],
                    "molecule_rank"
                ]
            else:
                rank = np.nan

        ranks.append(rank)

    metrics = evaluate_ranks(ranks)

    print("\nResults:")
    for metric, value in metrics.items():
        print(f"{metric:8s}: {value:.4f}")

    all_results.append({
        "top_n": "all" if top_n is None else top_n,
        **metrics,
    })

TOP N: 10

Results:
Top-1   : 0.7800
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8476
TOP N: 20

Results:
Top-1   : 0.8000
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8596
TOP N: 30

Results:
Top-1   : 0.8000
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8602
TOP N: 50

Results:
Top-1   : 0.8200
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8717
TOP N: None

Results:
Top-1   : 0.8200
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8717


In [35]:
pd.DataFrame(all_results)

,top_n,Top-1,Top-5,Top-25,MRR@25
0,10,0.78,0.94,0.96,0.847576
1,20,0.80,0.94,0.96,0.859583
2,30,0.80,0.94,0.96,0.860152
3,50,0.82,0.94,0.96,0.871667
4,all,0.82,0.94,0.96,0.871667


In [36]:
tolerance_results = []

for tolerance in [0.005, 0.0075, 0.01, 0.015, 0.02]:

    print("=" * 60)
    print("TOLERANCE:", tolerance)
    print("=" * 60)

    ranks = []

    for _, query in benchmark_queries.iterrows():

        results = retrieve_molecules_top_n(
            query,
            reference_df,
            top_n=None,
            precursor_tolerance=0.02,
            neutral_mass_tolerance=0.005,
            tolerance=tolerance,
        )

        if len(results) == 0:
            rank = np.nan

        else:
            matches = results.index[
                results["inchikey14"] == query["inchikey14"]
            ]

            if len(matches):
                rank = results.loc[
                    matches[0],
                    "molecule_rank"
                ]
            else:
                rank = np.nan

        ranks.append(rank)

    metrics = evaluate_ranks(ranks)

    print()

    for metric, value in metrics.items():
        print(f"{metric:8s}: {value:.4f}")

    tolerance_results.append({
        "tolerance": tolerance,
        **metrics,
    })

tolerance_results_df = pd.DataFrame(tolerance_results)

display(tolerance_results_df)

TOLERANCE: 0.005

Top-1   : 0.8200
Top-5   : 0.9600
Top-25  : 0.9600
MRR@25  : 0.8723
TOLERANCE: 0.0075

Top-1   : 0.8200
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8717
TOLERANCE: 0.01

Top-1   : 0.8200
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8717
TOLERANCE: 0.015

Top-1   : 0.8200
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8712
TOLERANCE: 0.02

Top-1   : 0.8200
Top-5   : 0.9400
Top-25  : 0.9600
MRR@25  : 0.8702


,tolerance,Top-1,Top-5,Top-25,MRR@25
0,0.0050,0.82,0.96,0.96,0.872333
1,0.0075,0.82,0.94,0.96,0.871667
2,0.0100,0.82,0.94,0.96,0.871667
3,0.0150,0.82,0.94,0.96,0.871190
4,0.0200,0.82,0.94,0.96,0.870152
